<a href="https://colab.research.google.com/github/sebace05/64610-Memory/blob/codex%2Fqwen-capability-checkpoint/notebooks/qwen_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen CookingWorld capability checkpoint

Run the repository code on a Colab GPU. This notebook is a launcher, not a second implementation. Start with an L4-class 24 GB GPU if available; use FP16 explicitly for a T4. No training or quantization is performed.

**Before running:** commit and push the implementation, then enter its full Git SHA below. A private repository needs your normal secure GitHub access in Colab; do not paste tokens into this notebook. The public Qwen checkpoint ordinarily needs no token.

This checkpoint retains all prior visible observations/actions. It is not one of the three 256-token memory conditions. Start with one development game and inspect its trace before running more seeds. Real Qwen gameplay has not been validated by the offline tests.

In [15]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import uuid

REPO_URL = "https://github.com/sebace05/64610-Memory.git"
REPO_REF = "985a05802dd9829893d0ddb1764ac9cbd7408a21"
REPO_DIR = Path("/content/64610-Memory-java-fix")
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = None  # Resolve once below; paste the recorded SHA here when freezing a protocol.
DTYPE = "float16"     # Explicitly change to "float16" for a T4; keep fixed within a pilot.
RECHECK_COST = 2
ACTION_BUDGET = 80
CONTEXT_LIMIT = 8192   # Overflow is an error; history is never silently truncated.
ACTION_MAX_NEW_TOKENS = 128
GENERATION_SEED = 0
FIRST_SEED = 0
MORE_SEEDS = [1, 2, 3, 4]  # Development only. Keep evaluation seeds separate.

## 1. Check out the selected source revision

An existing checkout is reused only if clean and already at the requested commit. No reset or overwrite is performed.

In [16]:
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    raise ValueError("Set REPO_REF to the full pushed implementation commit SHA first.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True)
status = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
source_sha = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if status or source_sha != REPO_REF:
    raise RuntimeError("Existing checkout is dirty or at a different revision; preserve it and use a fresh REPO_DIR.")
print("Source commit:", source_sha)

Source commit: 985a05802dd9829893d0ddb1764ac9cbd7408a21


## 2. Persist results before launching inference

Google Drive authorization happens through Colab. Every new launch directory preserves earlier attempts, including partial/error traces. Drive writes may add overhead to episode wall time; model latency is recorded separately.

In [17]:
from google.colab import drive

drive.mount("/content/drive")
PILOT_DIR = Path("/content/drive/MyDrive/64610-Memory/qwen-pilots") / uuid.uuid4().hex
PILOT_DIR.mkdir(parents=True, exist_ok=False)
print("Persistent pilot directory:", PILOT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent pilot directory: /content/drive/MyDrive/64610-Memory/qwen-pilots/3e31088b2e1843af91d1cf11582a081c


## 3. Install an isolated inference environment and run tests

The separate environment avoids conflicts with Colab's preinstalled Transformers, torchvision, or audio packages. This setup downloads the pinned CUDA-enabled PyTorch distribution; it may take several minutes. No model weights are loaded yet.

In [20]:
import shutil

# Keep JVM container warnings off the stdout used by Py4J's port handshake.
# Set the parent environment too, so recreating ENV preserves this fix.
JAVA_GATEWAY_LOG_OPTIONS = "-Xlog:os+container=off:stdout -Xlog:os+container=warning:stderr"
java_options = os.environ.get("JAVA_TOOL_OPTIONS", "")
if JAVA_GATEWAY_LOG_OPTIONS not in java_options:
    java_options = (java_options + " " + JAVA_GATEWAY_LOG_OPTIONS).strip()
os.environ["JAVA_TOOL_OPTIONS"] = java_options

if shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "openjdk-17-jre-headless"], check=True)
subprocess.run(["java", "-version"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.11"], check=True)
runtime = Path("/content/memory-qwen-venv")
if not runtime.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", str(runtime)], check=True)
PYTHON = str(runtime / "bin/python")
ENV = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", HF_HOME="/content/hf-cache")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements-model.txt"], cwd=REPO_DIR, env=ENV, check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], cwd=REPO_DIR, env=ENV, check=True)

test_result = subprocess.run(
    [PYTHON, "-m", "pytest", "-q", "--tb=short"],
    cwd=REPO_DIR, env=ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
(PILOT_DIR / "pytest.log").write_text(test_result.stdout)
print(test_result.stdout)
print("Test log saved to:", PILOT_DIR / "pytest.log")
test_result.check_returncode()

(PILOT_DIR / "pip-freeze.txt").write_text(
    subprocess.check_output([PYTHON, "-m", "pip", "freeze"], env=ENV, text=True)
)


........................................................................ [ 41%]
........................................................................ [ 83%]
.............................                                            [100%]
173 passed in 44.84s

Test log saved to: /content/drive/MyDrive/64610-Memory/qwen-pilots/3e31088b2e1843af91d1cf11582a081c/pytest.log


1013

## 4. Verify the GPU and freeze the model revision

Use the selected precision explicitly; an unsupported GPU raises an error. No CPU fallback is used for this experiment. Copy the resolved model SHA into MODEL_REVISION for subsequent frozen runs.

In [21]:
gpu_check = """
import json, torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime."
if {dtype!r} == "bfloat16":
    assert torch.cuda.is_bf16_supported(), "Select FP16 explicitly for this GPU."
print(json.dumps({{"gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "cuda": torch.version.cuda}}))
""".format(dtype=DTYPE)
gpu_info = json.loads(subprocess.check_output([PYTHON, "-c", gpu_check], env=ENV, text=True))
print(gpu_info)
if MODEL_REVISION is None:
    resolve = "from huggingface_hub import HfApi; print(HfApi().model_info({!r}).sha)".format(MODEL_ID)
    MODEL_REVISION = subprocess.check_output([PYTHON, "-c", resolve], env=ENV, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION):
    raise ValueError("Expected an immutable model commit SHA.")
manifest = dict(source_commit=source_sha, model_id=MODEL_ID, model_revision=MODEL_REVISION,
                dtype=DTYPE, gpu=gpu_info, recheck_cost=RECHECK_COST, action_budget=ACTION_BUDGET,
                context_limit=CONTEXT_LIMIT, action_max_new_tokens=ACTION_MAX_NEW_TOKENS,
                generation_seed=GENERATION_SEED, first_seed=FIRST_SEED, more_seeds=MORE_SEEDS,
                strategy="full_history", diagnostic_only=True)
(PILOT_DIR / "pilot-manifest.json").write_text(json.dumps(manifest, indent=2))
print("Model commit:", MODEL_REVISION)

{'gpu': 'Tesla T4', 'torch': '2.8.0+cu128', 'cuda': '12.8'}
Model commit: cdbee75f17c01a7cc42f958dc650907174af0554


## 5. Run one real episode

Weights load once per invocation. A valid game loss returns exit code zero; model/context/infrastructure errors return nonzero and preserve their traces. Do not add constrained decoding or hidden history to improve this checkpoint without documenting the change.

In [22]:
def run_diagnostic(seeds):
    existing = [json.loads(p.read_text()) for p in PILOT_DIR.glob("*/result.json")]
    if any(r["config"]["seed"] in seeds for r in existing):
        raise RuntimeError("Requested seed already has a result. Preserve this attempt and start a new pilot directory to rerun.")
    if any(not (p.parent / "result.json").exists() for p in PILOT_DIR.glob("*/events.jsonl")):
        raise RuntimeError("This pilot contains a partial episode. Inspect it and use a separate directory for retries.")
    command = [PYTHON, "-m", "game_memory.run_episode", "--backend", "transformers",
               "--strategy", "full_history", "--model-id", MODEL_ID, "--model-revision", MODEL_REVISION,
               "--dtype", DTYPE, "--context-limit", str(CONTEXT_LIMIT),
               "--action-max-new-tokens", str(ACTION_MAX_NEW_TOKENS),
               "--generation-seed", str(GENERATION_SEED), "--recheck-cost", str(RECHECK_COST),
               "--action-budget", str(ACTION_BUDGET), "--output-dir", str(PILOT_DIR),
               "--seeds", *map(str, seeds)]
    subprocess.run(command, cwd=REPO_DIR, env=ENV, check=True)

run_diagnostic([FIRST_SEED])

## 6. Audit the checkpoint and inspect the trace

The report independently checks budgets, tokens, request/response pairing, labels, and completion. It keeps infrastructure errors separate from gameplay losses. A technically valid report does not demonstrate adequate gameplay capability. Open the saved events.jsonl to inspect exact prompts and decisions.

In [23]:
def audit_pilot():
    completed = subprocess.run([PYTHON, "-m", "game_memory.summarize_diagnostics", str(PILOT_DIR)],
                               cwd=REPO_DIR, env=ENV, text=True, capture_output=True)
    if completed.stdout:
        (PILOT_DIR / "checkpoint-report.json").write_text(completed.stdout)
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()

audit_pilot()

{
  "episodes": [
    {
      "action_repair_calls": 14,
      "budget_spent": 79,
      "directory": "4168f196f79e4a5db196c993cc071686",
      "episode_id": "4168f196f79e4a5db196c993cc071686",
      "error": null,
      "input_tokens": 87811,
      "invalid_actions": 1,
      "kind": "full_history_diagnostic",
      "model_elapsed_seconds": 142.60496001299907,
      "output_limit_hits": 0,
      "output_tokens": 527,
      "recipe_rechecks": 34,
      "runtime_seconds": 144.33364842000037,
      "seed": 0,
      "status": "completed",
      "steps": 46,
      "success": false,
      "termination_reason": "budget_exhaustion"
    }
  ],
  "fixture_episodes": 0,
  "infrastructure_checks_passed": true,
  "interpretation": "Inspect traces for gameplay, planning, and information-use failures. This report does not establish memory-strategy effectiveness or statistical significance.",
  "issues": [],
  "kind": "full_history_checkpoint_report",
  "logging_checks_passed": true,
  "pending": [],

In [24]:
from pathlib import Path
import json
import pandas as pd

episode_id = "4168f196f79e4a5db196c993cc071686"
root = Path("/content/drive/MyDrive/64610-Memory/qwen-pilots")
matches = list(root.glob(f"*/{episode_id}/events.jsonl"))

if len(matches) != 1:
    raise RuntimeError(f"Expected one trace; found: {matches}")

trace_path = matches[0]
events = [
    json.loads(line)
    for line in trace_path.read_text().splitlines()
]
print("Trace:", trace_path)

responses = {
    (e["step"], e["purpose"]): e["text"]
    for e in events
    if e["event"] == "model_response"
}

rows = []
for e in events:
    if e["event"] != "action":
        continue
    t = e["transition"]
    rows.append({
        "step": e["step"],
        "action": t["action"],
        "cost": t["action_cost"],
        "budget_left": t["view"]["budget_remaining"],
        "score": t["score"],
        "valid": t["valid_action"],
        "observation": t["view"]["observation"],
        "first_response": responses.get((e["step"], "action"), ""),
        "repair_response": responses.get((e["step"], "action_repair"), ""),
    })

df = pd.DataFrame(rows)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df)

print("\nAction frequencies:")
print(df["action"].value_counts())

Trace: /content/drive/MyDrive/64610-Memory/qwen-pilots/3e31088b2e1843af91d1cf11582a081c/4168f196f79e4a5db196c993cc071686/events.jsonl


,step,action,cost,budget_left,score,valid,observation,first_response,repair_response
0,1,read cookbook,2,78,0.00,True,Gather all following ingredients and follow th...,"{""action"": ""read cookbook""}",
1,2,examine fridge,1,77,0.00,True,a fridge that is closed,"{""action"": ""examine fridge""}",
2,3,open fridge,1,76,0.00,True,You open the fridge. The fridge contains some ...,"{""action"": ""open fridge""}",
3,4,take water,1,75,0.25,True,You take the water.,"{""action"": ""take water""}",
4,5,inventory,1,74,0.25,True,Inventory: \n some water\n,"{""action"": ""inventory""}",
5,6,examine kitchen cupboard,1,73,0.25,True,a kitchen cupboard that is closed,"{""action"": ""examine kitchen cupboard""}",
6,7,open kitchen cupboard,1,72,0.25,True,You open the kitchen cupboard. It's empty inside.,"{""action"": ""open kitchen cupboard""}",
7,8,put water in counter,1,71,0.25,True,You put the water in the counter.,"{""action"": ""put water in counter""}",
8,9,examine cookbook,2,69,0.25,True,Gather all following ingredients and follow th...,"{""action"": ""examine cookbook""}",
9,10,examine cutlery drawer,1,68,0.25,True,a cutlery drawer that is closed,"{""action"": ""examine cutlery drawer""}",



Action frequencies:
action
examine cookbook                                    34
examine fridge                                       2
open fridge                                          1
read cookbook                                        1
take water                                           1
inventory                                            1
open kitchen cupboard                                1
examine kitchen cupboard                             1
put water in counter                                 1
examine cutlery drawer                               1
open cutlery drawer                                  1
[invalid_action_output] {"action": "take flour"}     1
Name: count, dtype: int64


In [25]:
STEP = 1

for e in events:
    if e.get("step") != STEP:
        continue

    if e["event"] == "model_request":
        print("\nREQUEST:", e["purpose"])
        for message in e["messages"]:
            print(f"\n{message['role'].upper()}:\n{message['content']}")

    elif e["event"] == "model_response":
        print("\nRESPONSE:", e["purpose"])
        print(e["text"])

    elif e["event"] == "action":
        print("\nGAME TRANSITION:")
        print(json.dumps(e["transition"], indent=2))


REQUEST: action

SYSTEM:
Choose one permitted game action using only the current observation and visible interaction history. Recipe rereads consume the stated budget. Return only a JSON object of the form {"action": "exact permitted action"}.

USER:
Current game input:
{"allowed_actions": ["examine cookbook", "examine counter", "examine cutlery drawer", "examine dining chair", "examine dishwasher", "examine fridge", "examine kitchen cupboard", "examine knife", "examine oven", "examine stove", "examine trash can", "inventory", "look around", "move east", "move south", "open cutlery drawer", "open dishwasher", "open fridge", "open kitchen cupboard", "open trash can", "read cookbook", "take cookbook", "take knife"], "budget_remaining": 80, "goal": "You are hungry! Let's cook a delicious meal. Check the cookbook in the kitchen for the recipe. Once done, enjoy your meal!", "observation": "You are in the kitchen. In one part of the room you see a stove. There is also an oven. You also see 

In [26]:
from google.colab import files
files.download(str(trace_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Optional: run four more development games

Run this cell after inspecting the first episode. Keep settings unchanged. The four games share one model load. Look for evidence of valid gameplay, planning ability, and use of information already present in history. Five games provide a feasibility diagnostic, not a reliable success-rate estimate. Repeated full-history failure should be diagnosed before attributing bounded-memory losses to forgetting.

In [ ]:
run_diagnostic(MORE_SEEDS)
audit_pilot()

## Decision and next milestone

- Fix infrastructure or unexplained accounting failures before proceeding.
- Inspect formatting failures, output-limit hits, loops, navigation/cooking mistakes, and budget exhaustion.
- If capability looks promising, implement and test the real prose/structured/hybrid writers and the six-condition development pilot.
- If gameplay fails with history, use development data to diagnose capability or adjust task difficulty. Record every change and rerun consistently.
- Keep held-out evaluation seeds untouched. Do not interpret fewer rereads alone as better memory.

Results, the model/source revisions, dependency freeze, and reports are in PILOT_DIR. Disconnect the GPU runtime when finished. This notebook has not been executed on Colab as part of the CPU-only implementation validation.